In [1]:
import pandas as pd

final_df=pd.read_csv("HAPPY_POPULATION.csv")

In [2]:
final_df.head()

,happiness_rank,year,country,region,happiness_score,gdp_percapita,life_expectancy,family,freedom,generosity,government_corruption,total_pop,pop_density,age_pop
0,1,2015,Switzerland,Western Europe,7.587,1.39651,0.94143,1.34951,0.66557,0.29678,0.41978,8283934.0,207.139778,41.259268
1,2,2015,Iceland,Western Europe,7.561,1.30232,0.94784,1.40223,0.62877,0.43630,0.14145,330951.5,3.301492,34.940392
2,3,2015,Denmark,Western Europe,7.527,1.32548,0.87464,1.36058,0.64938,0.34139,0.48357,5683701.0,134.068524,40.497999
3,4,2015,Norway,Western Europe,7.522,1.45900,0.88521,1.33095,0.66973,0.34699,0.36503,5189771.0,17.058933,38.176429
4,5,2015,Canada,North America,7.427,1.32629,0.90563,1.32261,0.63297,0.45811,0.32957,35962234.0,3.954716,39.629038


<StringArray>
[                 'Western Europe',                   'North America',
       'Australia and New Zealand', 'Middle East and Northern Africa',
     'Latin America and Caribbean',               'Southeastern Asia',
      'Central and Eastern Europe',                    'Eastern Asia',
              'Sub-Saharan Africa',                   'Southern Asia']
Length: 10, dtype: str

In [6]:
# Creating the REGION df to be imported to SQL:

region_names = final_df["region"].unique()

region_df = pd.DataFrame({"region_id": range(1, len(region_names) + 1),"region_name": region_names})
region_df

,region_id,region_name
0,1,Western Europe
1,2,North America
2,3,Australia and New Zealand
3,4,Middle East and Northern Africa
4,5,Latin America and Caribbean
5,6,Southeastern Asia
6,7,Central and Eastern Europe
7,8,Eastern Asia
8,9,Sub-Saharan Africa
9,10,Southern Asia


In [8]:
# Creating the YEAR df to be imported to SQL:

years = final_df["year"].unique()

year_df = pd.DataFrame({"year_id": range(1, len(years) + 1),"year": years})
year_df

,year_id,year
0,1,2015
1,2,2016
2,3,2017
3,4,2018
4,5,2019


In [10]:
# Creating the COUNTRY df to be imported to SQL:

countries = final_df["country"].unique()
region_map = dict(zip(region_df["region_name"], region_df["region_id"]))
country_regions = (final_df.drop_duplicates("country").set_index("country")["region"].loc[countries].map(region_map).values)

country_df = pd.DataFrame({"country_id": range(1, len(countries) + 1),"country_name": countries, "region_id":country_regions})

country_df

,country_id,country_name,region_id
0,1,Switzerland,1
1,2,Iceland,1
2,3,Denmark,1
3,4,Norway,1
4,5,Canada,2
...,...,...,...
161,162,Somaliland Region,9
162,163,Namibia,9
163,164,South Sudan,9
164,165,North Macedonia,7


In [11]:
# Creating year_country df to bridge data from tables country and years as they have a relationship of many to many:

country_map = dict(zip(country_df["country_name"], country_df["country_id"]))
year_map = dict(zip(year_df["year"], year_df["year_id"]))

year_country_df = final_df[["country","year","total_pop","pop_density","age_pop"]].drop_duplicates()

year_country_df = pd.DataFrame({"year_country_id": range(1, len(year_country_df) + 1),
    "country_id": year_country_df["country"].map(country_map),
    "year_id": year_country_df["year"].map(year_map),
    "total_pop": year_country_df["total_pop"].values,
    "pop_density": year_country_df["pop_density"].values,
    "age_pop": year_country_df["age_pop"].values})

year_country_df

,year_country_id,country_id,year_id,total_pop,pop_density,age_pop
0,1,1,1,8283934.0,207.139778,41.259268
1,2,2,1,330951.5,3.301492,34.940392
2,3,3,1,5683701.0,134.068524,40.497999
3,4,4,1,5189771.0,17.058933,38.176429
4,5,5,1,35962234.0,3.954716,39.629038
...,...,...,...,...,...,...
777,776,154,5,12776103.0,527.720074,18.741527
778,777,146,5,59174891.0,66.803670,16.833554
779,778,153,5,37856121.0,58.303333,16.234260
780,779,148,5,4944703.5,7.937179,14.367738


In [12]:
year_country_df.isna().sum()

year_country_id    0
country_id         0
year_id            0
total_pop          0
pop_density        0
age_pop            0
dtype: int64

In [ ]:
# 
year_country_map = dict(zip(zip(year_country_df["country_id"], year_country_df["year_id"]),year_country_df["year_country_id"]))
country_map = dict(zip(country_df["country_name"], country_df["country_id"]))
year_map = dict(zip(year_df["year"], year_df["year_id"]))
